# Battle Lab · Fase 2 — VGC-Bench sin piedad

Esta es la libreta canónica del Battle Lab. Prepara un servidor privado de Pokémon Showdown, valida dos equipos Champions M-C y enfrenta el checkpoint público final de **VGC-Bench contra sí mismo**. La red decide tanto el Team Preview como cada turno.

**GPU recomendada, no obligatoria.** Con `DEVICE = "auto"` se usa CUDA cuando Colab la ofrece y CPU en caso contrario. Las decisiones son deterministas (máxima probabilidad legal), y el resumen/replays se guardan en Google Drive.

> Nota honesta: el checkpoint fue entrenado con equipos M-A/M-B. Aquí lo evaluamos en M-C sin reentrenarlo, así que será mucho más serio que los bots del smoke test, pero no asumimos un Elo competitivo hasta medirlo.


## 1. Configuración

Para la prueba normal solo hace falta cambiar `BATTLES`. `DEVICE = "auto"` exprime la GPU si está disponible. `PKMN_REF` permite probar una rama o commit concreto antes de integrarlo a `main`.


In [ ]:
PKMN_REPOSITORY = "https://github.com/Iesyo/pkmn.git"  # @param {type:"string"}
PKMN_REF = "main"  # @param {type:"string"}
BATTLES = 20  # @param {type:"integer"}
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
SEED = 260913  # @param {type:"integer"}
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_RESULTS = "Pokemon VGC/BattleLab/results/phase-2"  # @param {type:"string"}
RUNTIME_ROOT = "/content/battle-lab-runtime"
PKMN_ROOT = "/content/pkmn"
NODE_VERSION = "24.21.0"

if BATTLES < 1:
    raise ValueError("BATTLES debe ser mayor que cero")


## 2. Preparar el código y las dependencias


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time

pkmn_root = Path(PKMN_ROOT)
runtime_root = Path(RUNTIME_ROOT)
packages_root = runtime_root / "python-packages"
battle_lab_python = Path(sys.executable)
battle_lab_env = os.environ.copy()
previous_pythonpath = battle_lab_env.get("PYTHONPATH")
battle_lab_env["PYTHONPATH"] = str(packages_root) + (os.pathsep + previous_pythonpath if previous_pythonpath else "")
started = time.monotonic()
prep_total = 6

def prep_progress(completed, detail):
    elapsed = time.monotonic() - started
    eta = elapsed / completed * (prep_total - completed) if completed else None
    filled = round(24 * completed / prep_total)
    bar = "█" * filled + "░" * (24 - filled)
    eta_text = f"{eta:.0f}s" if eta is not None else "calculando"
    print(f"Preparación [{bar}] {completed}/{prep_total} · {elapsed:.0f}s · ETA {eta_text} · {detail}", flush=True)

def run_live(command, *, cwd=None, label, attempts=1, env=None):
    command = [str(part) for part in command]
    for attempt in range(1, attempts + 1):
        print(f"\n▶ {label} · intento {attempt}/{attempts}", flush=True)
        step_started = time.monotonic()
        process = subprocess.Popen(
            command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        lines = []
        assert process.stdout is not None
        for line in process.stdout:
            lines.append(line)
            print(line, end="", flush=True)
        return_code = process.wait()
        if return_code == 0:
            print(f"✅ {label} ({time.monotonic() - step_started:.1f}s)", flush=True)
            return
        if attempt < attempts:
            wait_seconds = 5 * attempt
            print(f"⚠️ {label} falló; reintento en {wait_seconds}s.", flush=True)
            time.sleep(wait_seconds)
            continue
        tail = "".join(lines[-60:]) or "<el comando no produjo salida>"
        raise RuntimeError(f"{label} falló tras {attempts} intento(s).\nÚltimas líneas:\n{tail}")

prep_progress(0, f"Python {sys.version.split()[0]}")
if not (pkmn_root / ".git").is_dir():
    run_live(
        ["git", "clone", "--filter=blob:none", "--no-checkout", PKMN_REPOSITORY, pkmn_root],
        label="Clonar pkmn",
    )
else:
    dirty = subprocess.run(
        ["git", "status", "--porcelain", "--untracked-files=no"],
        cwd=pkmn_root, text=True, capture_output=True, check=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError("El checkout temporal de pkmn contiene cambios; usa un runtime limpio.")

run_live(["git", "fetch", "--depth", "1", "origin", PKMN_REF], cwd=pkmn_root, label="Actualizar pkmn", attempts=3)
run_live(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=pkmn_root, label="Fijar revisión de pkmn")
prep_progress(1, "Código fijado")

node_result = subprocess.run(["node", "--version"], text=True, capture_output=True) if shutil.which("node") else None
node_major = int(node_result.stdout.strip().lstrip("v").split(".")[0]) if node_result and node_result.returncode == 0 else 0
if node_major == 0 or not shutil.which("npm"):
    if not shutil.which("apt-get"):
        raise RuntimeError("Este runtime no incluye Node.js/npm ni apt-get para preparar el instalador.")
    apt_env = os.environ.copy()
    apt_env["DEBIAN_FRONTEND"] = "noninteractive"
    run_live(["apt-get", "update"], label="Actualizar catálogo de Node.js", attempts=3, env=apt_env)
    run_live(["apt-get", "install", "-y", "nodejs", "npm"], label="Preparar instalador de Node.js", attempts=3, env=apt_env)
if node_major < 24:
    run_live(["npm", "install", "--global", "n@latest"], label="Instalar gestor de Node.js", attempts=3)
    run_live(["n", NODE_VERSION], label=f"Instalar Node.js {NODE_VERSION}", attempts=3)
run_live(["node", "--version"], label="Verificar Node.js")
run_live(["npm", "--version"], label="Verificar npm")
node_major = int(subprocess.check_output(["node", "--version"], text=True).strip().lstrip("v").split(".")[0])
if node_major < 24:
    raise RuntimeError(f"Battle Lab requiere Node.js >=24; el runtime todavía tiene {node_major}.")
prep_progress(2, f"Node.js y npm listos · {NODE_VERSION}")

packages_root.mkdir(parents=True, exist_ok=True)
prep_progress(3, "Directorio aislado listo")

run_live(
    [battle_lab_python, "-m", "pip", "install", "--target", packages_root, "--upgrade", "--retries", "10", "--timeout", "120", "-r", "battle_lab/requirements-phase1.txt"],
    cwd=pkmn_root, label="Instalar dependencias base del Battle Lab", attempts=3,
)
prep_progress(4, "Dependencias base instaladas")

run_live(
    [battle_lab_python, "-m", "pip", "install", "--target", packages_root, "--upgrade", "--no-deps", "--retries", "10", "--timeout", "120", "-r", "battle_lab/requirements-phase2.txt"],
    cwd=pkmn_root, label="Instalar inferencia VGC-Bench sin duplicar PyTorch/CUDA", attempts=3,
)
prep_progress(5, "VGC-Bench listo")

run_live(
    [battle_lab_python, "-c", "import poke_env, psutil, torch, stable_baselines3; print(f'VGC-Bench listo · torch {torch.__version__} · CUDA {torch.cuda.is_available()} · SB3 {stable_baselines3.__version__}')"],
    label="Verificar dependencias", env=battle_lab_env,
)
prep_progress(6, f"Todo listo en {time.monotonic() - started:.1f}s")


## 3. Conectar Google Drive

Solo se guardarán el JSON final y un ZIP de replays/logs. Showdown, el checkout fijado de VGC-Bench y el checkpoint permanecen en el disco rápido de Colab.


In [ ]:
from pathlib import Path

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    results_dir = Path("/content/drive/MyDrive") / DRIVE_RESULTS
else:
    results_dir = Path(RUNTIME_ROOT) / "results"

results_dir.mkdir(parents=True, exist_ok=True)
print(f"Resultados: {results_dir}")


## 4. Ejecutar VGC-Bench vs. VGC-Bench en M-C


In [ ]:
command = [
    str(battle_lab_python),
    "battle_lab/vgc_bench_battle.py",
    "--runtime-root", RUNTIME_ROOT,
    "--results-dir", str(results_dir),
    "--battles", str(BATTLES),
    "--device", DEVICE,
    "--seed", str(SEED),
]
run_live(command, cwd=pkmn_root, label="Soltar a VGC-Bench sin piedad", env=battle_lab_env)


## 5. Resumen verificable


In [ ]:
import json
from IPython.display import HTML, display

result_files = sorted(results_dir.glob("vgc-bench-*.json"), key=lambda path: path.stat().st_mtime)
if not result_files:
    raise RuntimeError("No se encontró el resultado de VGC-Bench")

latest_result = result_files[-1]
result = json.loads(latest_result.read_text(encoding="utf-8"))
battles = result["battles"]
showdown = result["showdown"]
vgc = result["vgcBench"]
policy = vgc["policy"]
wins = battles["wins"]
display(HTML(f"""
<div style='padding:18px;border-radius:14px;background:#07111f;color:#e2e8f0;border:1px solid #164e63'>
  <div style='font-size:20px;font-weight:800;color:#67e8f9'>⚔️ VGC-Bench jugó sin piedad</div>
  <div style='margin-top:10px'>Formato: <b>{showdown['format']}</b></div>
  <div>Showdown: <code>{showdown['commit'][:12]}</code></div>
  <div>VGC-Bench: <code>{vgc['commit'][:12]}</code> · checkpoint <code>{vgc['checkpoint']['sha256'][:12]}</code></div>
  <div>Política: <b>determinista</b> · Team Preview IA · dispositivo <b>{policy['device']}</b></div>
  <div>Combates: <b>{battles['completed']}/{battles['requested']}</b></div>
  <div>Victorias: Alpha <b>{wins['alpha']}</b> · Beta <b>{wins['beta']}</b> · Empates <b>{wins['ties']}</b></div>
  <div>Rendimiento: <b>{battles['battlesPerMinute']} combates/min</b></div>
  <div style='margin-top:10px;color:#94a3b8'>JSON: {latest_result}</div>
  <div style='color:#94a3b8'>Replays: {result['artifacts']['replaysZip']}</div>
</div>
"""))
